# Detección de grietas mediante aprendizaje profundo

Este notebook prepara el dataset, lo organiza para entrenamiento, entrena un modelo de clasificación binaria para distinguir entre imágenes con grietas y sin grietas, y evalúa sus resultados.

## Objetivo
- Preparar los datos de forma reproducible
- Crear particiones de entrenamiento/validación/prueba
- Entrenar un modelo ligero y efectivo con transfer learning
- Guardar el modelo y las métricas finales


In [ ]:
# Instalación de dependencias
!apt-get -qq update >/dev/null
!apt-get -qq install unrar >/dev/null

# Verificación de versiones
import sys
print(sys.version)


In [ ]:
from pathlib import Path
import os
import shutil
import zipfile
import random
import json

import numpy as np
import tensorflow as tf

from google.colab import drive, files
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

ROOT = Path('/content')
ZIP_PATH = ROOT / 'grietas_dataset.zip'
DRIVE_DATASET = Path('/content/drive/MyDrive/grietas_dataset.zip')
WORK_DIR = ROOT / 'dataset_real'
TARGET_DATASET = ROOT / 'dataset'

print('TensorFlow:', tf.__version__)


In [ ]:
# Montar Google Drive si existe el archivo allí
try:
    drive.mount('/content/drive')
except Exception as e:
    print('Drive no disponible:', e)

# Intentar encontrar el ZIP en Drive o permitir subirlo manualmente
if DRIVE_DATASET.exists():
    print(f'Archivo encontrado en Drive: {DRIVE_DATASET}')
    shutil.copy2(DRIVE_DATASET, ZIP_PATH)
elif ZIP_PATH.exists():
    print(f'Archivo encontrado en /content: {ZIP_PATH}')
else:
    print('No se encontró el archivo en Drive. Sube el ZIP desde el siguiente cuadro.')
    uploaded = files.upload()
    for name, data in uploaded.items():
        ZIP_PATH.write_bytes(data)

print('ZIP listo en:', ZIP_PATH)


In [ ]:
def extract_archive(zip_path: Path, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(output_dir)

    rar_files = sorted(output_dir.rglob('*.rar'))
    if not rar_files:
        print('No se encontraron archivos .rar dentro del ZIP.')
        return

    for rar_file in rar_files:
        print(f'Extrayendo {rar_file.name}...')
        os.system(f'unrar x "{rar_file}" "{output_dir}" > /dev/null 2>&1')

    print('Extracción completada.')

extract_archive(ZIP_PATH, WORK_DIR)
print('Contenido del directorio raiz:')
for item in sorted(WORK_DIR.iterdir())[:10]:
    print(item.name)


In [ ]:
# Inspección del dataset
def list_directories(path: Path):
    return sorted([p for p in path.iterdir() if p.is_dir()])

dirs = list_directories(WORK_DIR)
print('Directorios encontrados en la raíz del dataset:')
for d in dirs:
    print('-', d.name)

if dirs:
    nested = list_directories(dirs[0])
    print(f'
Primer nivel dentro de {dirs[0].name}:')
    for d in nested:
        print('-', d.name)
        print('  Ejemplos:', sorted([p.name for p in d.iterdir()])[:3])


In [ ]:
def detect_class_dirs(dataset_root: Path):
    possible = []
    for d in sorted(dataset_root.rglob('*')):
    if d.is_dir() and any(img.suffix.lower() in {'.jpg', '.jpeg', '.png'} for img in d.iterdir()):
        possible.append(d)

    class_dirs = []
    for d in possible:
        name = d.name.lower()
        if 'neg' in name or 'no' in name or 'clean' in name or 'without' in name:
            class_dirs.append(('no_crack', d))
        elif 'pos' in name or 'crack' in name or 'with' in name or 'yes' in name:
            class_dirs.append(('crack', d))

    if not class_dirs:
        raise ValueError('No se pudieron detectar carpetas de clases. Revisa la estructura del dataset.')

    return dict(class_dirs)

class_map = detect_class_dirs(WORK_DIR)
print('Clases detectadas:')
for label, path in class_map.items():
    print(f'- {label}: {path}')

labels = sorted(class_map.keys())
print(f'Labels finales: {labels}')


In [ ]:
def build_dataset_structure(root: Path, class_map: dict, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15):
    if abs(train_ratio + val_ratio + test_ratio - 1.0) > 1e-6:
        raise ValueError('Las proporciones deben sumar 1.0')

    target_root = root / 'dataset'
    if target_root.exists():
        shutil.rmtree(target_root)
    target_root.mkdir(parents=True, exist_ok=True)

    for split in ['train', 'val', 'test']:
        for label in ['crack', 'no_crack']:
            (target_root / split / label).mkdir(parents=True, exist_ok=True)

    for label_name, source_dir in class_map.items():
        images = sorted([p for p in source_dir.iterdir() if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
        if not images:
            continue

        target_label = 'crack' if 'crack' in label_name.lower() or 'pos' in label_name.lower() else 'no_crack'
        train_images, temp_images = train_test_split(images, train_size=train_ratio, random_state=SEED, shuffle=True)
        val_images, test_images = train_test_split(temp_images, train_size=val_ratio / (val_ratio + test_ratio), random_state=SEED, shuffle=True)

        for img in train_images:
            dest = target_root / 'train' / target_label / img.name
            shutil.copy2(img, dest)
        for img in val_images:
            dest = target_root / 'val' / target_label / img.name
            shutil.copy2(img, dest)
        for img in test_images:
            dest = target_root / 'test' / target_label / img.name
            shutil.copy2(img, dest)

    print('Estructura creada en:', target_root)
    for split in ['train', 'val', 'test']:
        for label in ['crack', 'no_crack']:
            count = len(list((target_root / split / label).glob('*.*')))
            print(f'{split}/{label}: {count}')

build_dataset_structure(ROOT, class_map)


In [ ]:
TRAIN_DIR = ROOT / 'dataset' / 'train'
VAL_DIR = ROOT / 'dataset' / 'val'
TEST_DIR = ROOT / 'dataset' / 'test'

BATCH_SIZE = 32
IMG_SIZE = (224, 224)

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_test_gen = ImageDataGenerator(rescale=1./255)

train_ds = train_gen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True
)

val_ds = val_test_gen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

test_ds = val_test_gen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print(train_ds.class_indices)
print('Entrenamiento:', train_ds.samples)
print('Validación:', val_ds.samples)
print('Prueba:', test_ds.samples)


In [ ]:
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [ ]:
EPOCHS = 10
CALLBACKS = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('/content/best_model.keras', save_best_only=True, monitor='val_loss')
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=CALLBACKS,
    verbose=1
)

print('Entrenamiento finalizado')


In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f'Loss de prueba: {test_loss:.4f}')
print(f'Accuracy de prueba: {test_acc:.4f}')

y_true = test_ds.labels
y_pred = (model.predict(test_ds, verbose=0) >= 0.5).astype(int)

print('
Reporte de clasificación:')
print(classification_report(y_true, y_pred, target_names=['no_crack', 'crack']))

print('
Matriz de confusión:')
print(confusion_matrix(y_true, y_pred))

with open('/content/history.json', 'w') as f:
    json.dump(history.history, f)

model.save('/content/crack_model.keras')
print('Modelo guardado en /content/crack_model.keras')


## Conclusión

Este notebook ya no solo extrae el dataset, sino que deja un flujo completo de trabajo para entrenar un clasificador de grietas: preparación del dataset, particiones, data augmentation, transfer learning, evaluación y guardado del modelo.

### Siguientes pasos recomendados
- probar con otra arquitectura (ResNet50, EfficientNet)
- ajustar learning rate y epochs
- usar un conjunto de validación más equilibrado
- exportar el modelo a TensorFlow Lite o ONNX
- evaluar con imágenes reales del entorno de construcción
